# Day 025 Project Solution — AI Email-Responder Drafter

An `EmailDrafter` that parses incoming email, drafts a reply with the LLM, and returns a properly threaded `EmailMessage` ready for SMTP delivery.

In [ ]:
import email
import email.message
import ollama


def build_email_message(
    to: str,
    subject: str,
    body: str,
    from_addr: str = "sender@example.com",
) -> email.message.EmailMessage:
    msg = email.message.EmailMessage()
    msg["From"] = from_addr
    msg["To"] = to
    msg["Subject"] = subject
    msg.set_content(body)
    return msg


def parse_email_string(raw_email: str) -> email.message.Message:
    return email.message_from_string(raw_email)


def get_email_body(msg: email.message.Message) -> str:
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                raw = part.get_payload(decode=True)
                if raw is not None:
                    charset = part.get_content_charset() or "utf-8"
                    return raw.decode(charset, errors="replace")
                return str(part.get_payload() or "")
    raw = msg.get_payload(decode=True)
    if raw is not None:
        charset = msg.get_content_charset() or "utf-8"
        return raw.decode(charset, errors="replace")
    return str(msg.get_payload() or "")


def draft_reply(
    original_subject: str,
    original_body: str,
    context: str,
    model: str = "llama3.2",
) -> str:
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional email assistant. "
                    "Write concise, polite email replies. "
                    "Match the tone of the original. "
                    "Return only the email body — no subject line, no 'Subject:' prefix."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Original subject: {original_subject}\n\n"
                    f"Original message:\n{original_body}\n\n"
                    f"Context for your reply: {context}\n\n"
                    "Write a reply email body:"
                ),
            },
        ],
    )
    return response["message"]["content"]


def build_reply_email(
    original_msg: email.message.Message,
    reply_body: str,
    from_addr: str,
    to_addr: str | None = None,
) -> email.message.EmailMessage:
    reply = email.message.EmailMessage()
    subject = original_msg.get("Subject", "")
    if not subject.startswith("Re:"):
        subject = f"Re: {subject}"
    reply["Subject"] = subject
    reply["From"] = from_addr
    reply["To"] = to_addr or original_msg.get("From", "")
    msg_id = original_msg.get("Message-ID")
    if msg_id:
        reply["In-Reply-To"] = msg_id
    reply.set_content(reply_body)
    return reply


class EmailDrafter:
    def parse(self, raw_email: str) -> email.message.Message:
        return parse_email_string(raw_email)

    def draft(
        self,
        msg: email.message.Message,
        context: str,
        model: str = "llama3.2",
    ) -> str:
        body = get_email_body(msg)
        return draft_reply(
            original_subject=msg.get("Subject", ""),
            original_body=body,
            context=context,
            model=model,
        )

    def build_reply(
        self,
        original_msg: email.message.Message,
        reply_body: str,
        from_addr: str,
    ) -> email.message.EmailMessage:
        return build_reply_email(original_msg, reply_body, from_addr=from_addr)

    def run_pipeline(
        self,
        raw_email: str,
        context: str,
        from_addr: str,
        model: str = "llama3.2",
    ) -> dict:
        parsed = self.parse(raw_email)
        reply_body = self.draft(parsed, context=context, model=model)
        reply_msg = self.build_reply(parsed, reply_body, from_addr=from_addr)
        return {"parsed": parsed, "reply_body": reply_body, "reply_msg": reply_msg}

In [ ]:
SAMPLE_EMAIL = 'From: alice@example.com\nTo: bob@example.com\nSubject: Project Kickoff Meeting\nMessage-ID: <20260717090000.alice@example.com>\nDate: Mon, 17 Jul 2026 09:00:00 +0000\nContent-Type: text/plain; charset=utf-8\n\nHi Bob,\n\nCan we schedule a kickoff meeting for the new project?\nI am available Tuesday or Wednesday afternoon.\n\nBest,\nAlice\n'

## Action 1 — Parse the Incoming Email

In [ ]:
drafter = EmailDrafter()
parsed = drafter.parse(SAMPLE_EMAIL)
print('Parsed email:')
print(f"  From:    {parsed['From']}")
print(f"  To:      {parsed['To']}")
print(f"  Subject: {parsed['Subject']}")
body = get_email_body(parsed)
print(f"  Body ({len(body)} chars): {body[:80].strip()!r}")

## Action 2 — Draft an AI Reply

In [ ]:
reply_body = drafter.draft(
    parsed,
    context='Accept the meeting, propose Thursday at 2pm.',
)
print('\nDrafted reply:')
print(reply_body)

## Action 3 — Build and Inspect the Reply Email

In [ ]:
reply_msg = drafter.build_reply(parsed, reply_body, from_addr='bob@example.com')
print('\nReply email headers:')
print(f"  From:         {reply_msg['From']}")
print(f"  To:           {reply_msg['To']}")
print(f"  Subject:      {reply_msg['Subject']}")
print(f"  In-Reply-To:  {reply_msg.get('In-Reply-To')}")
print('\nDrafting complete!')